# DeepFace ライブラリによる顔の類似度ヒートマップの作成例

This Person Does Not Exist から取得した人物の顔写真から、類似度を計算してヒートマップ化

## 注意

ローカルで ipynb を実行するために ipykernel のインストールが必要です。
```
$ pip install ipykernel
```
を実行してからノートを実行してください。

In [ ]:
# パッケージをインストール
!pip install deepface tf_keras matplotlib seaborn pillow

In [ ]:
# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

# FaceNet の重みファイルを DeepFace の重みファイルディレクトリにコピー
!mkdir -p /root/.deepface/weights
!cp /content/drive/MyDrive/オープンキャンパス_2026_spring/演習1/facenet_weights.h5 /root/.deepface/weights/

In [ ]:
# 人物画像をダウンロード
import requests

url = "https://thispersondoesnotexist.com/"
num_images = 6

for i in range(1, num_images + 1):
    file_name = f"person_{i}.jpg"
    try:
        print(f"{i}枚目: '{url}' から画像をダウンロードしています...")
        response = requests.get(url)
        response.raise_for_status()
        with open(file_name, 'wb') as f:
            f.write(response.content)
        print(f"画像 '{file_name}' のダウンロードが完了しました。")
    except requests.exceptions.RequestException as e:
        print(f"{i}枚目の画像のダウンロードに失敗しました: {e}")


In [ ]:
# 顔画像の類似度を計算

import glob

# カレントディレクトリの画像ファイル一覧を取得
images = sorted(glob.glob("*.jpg")+glob.glob("*.jpeg")+glob.glob("*.png"))

from deepface import DeepFace
import numpy as np


# 類似度行列を計算
num = len(images)
similarity_matrix = np.zeros((num, num))


for i in range(num):
    for j in range(num):
        if i == j:
            similarity_matrix[i, j] = 1.0
        elif i < j:
            # 顔画像間の類似度を計算
            print(f"Calculating similarity between {images[i]} and {images[j]}...")
            result = DeepFace.verify(img1_path=images[i], img2_path=images[j],model_name='Facenet', enforce_detection=True)
            similarity = abs(1 - result['distance'])  # 距離が小さいほど類似
            similarity_matrix[i, j] = similarity
            similarity_matrix[j, i] = similarity


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

thumbnails = []
for img_path in images:
    img = Image.open(img_path)
    img.thumbnail((32, 32)) # Adjust thumbnail size as needed
    thumbnails.append(img)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(similarity_matrix, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Similarity')

# サムネイルをx軸とy軸のticksに表示
for idx, thumbnail in enumerate(thumbnails):
    imagebox = OffsetImage(thumbnail, zoom=1)
    ab = AnnotationBbox(imagebox, (idx, -0.5), frameon=False, box_alignment=(0.5, 0))
    ax.add_artist(ab)
    ab = AnnotationBbox(imagebox, (-0.5, idx), frameon=False, box_alignment=(1, 0.5))
    ax.add_artist(ab)

ax.set_xticks(range(num))
ax.set_yticks(range(num))
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_title("Similarity Matrix of Faces", pad=45)

# 類似度値を各セルに表示し、白い背景を追加
for i in range(num):
    for j in range(num):
        ax.text(j, i, f"{similarity_matrix[i, j]:.2f}",
                ha="center", va="center", color="black",
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'),
                fontsize=13) # フォントサイズを調整

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.lines as lines # Line2Dをインポート

# 類似度ペアとそのインデックスを収集
pairs_with_similarity = []
for i in range(num):
    for j in range(i + 1, num):
        pairs_with_similarity.append((similarity_matrix[i, j], i, j))

# 類似度でソート（降順）
sorted_pairs_desc = sorted(pairs_with_similarity, key=lambda x: x[0], reverse=True)
# 類似度でソート（昇順）
sorted_pairs_asc = sorted(pairs_with_similarity, key=lambda x: x[0])

top_3_pairs = sorted_pairs_desc[:3]
worst_3_pairs = sorted_pairs_asc[:3]

fig, axes = plt.subplots(3, 4, figsize=(16, 12)) # 3行、4列のサブプロット

fig.suptitle("Top 3 Most Similar Pairs (Left) vs. Worst 3 Similar Pairs (Right)", fontsize=16)

for i in range(3):
    # 類似度が高いペア (左側)
    sim_top, idx1_top, idx2_top = top_3_pairs[i]
    img1_top = Image.open(images[idx1_top])
    img2_top = Image.open(images[idx2_top])

    # Topペアの最初の画像
    axes[i, 0].imshow(img1_top)
    axes[i, 0].set_title(f"Top {i+1}: {images[idx1_top]}", fontsize=10)
    axes[i, 0].axis('off')

    # Topペアの2番目の画像と類似度スコア
    axes[i, 1].imshow(img2_top)
    axes[i, 1].set_title(f"{images[idx2_top]} (Similarity: {sim_top:.2f})", fontsize=10) # 画像名を追加
    axes[i, 1].axis('off')

    # 類似度が低いペア (右側)
    sim_worst, idx1_worst, idx2_worst = worst_3_pairs[i]
    img1_worst = Image.open(images[idx1_worst])
    img2_worst = Image.open(images[idx2_worst])

    # Worstペアの最初の画像
    axes[i, 2].imshow(img1_worst)
    axes[i, 2].set_title(f"Worst {i+1}: {images[idx1_worst]}", fontsize=10)
    axes[i, 2].axis('off')

    # Worstペアの2番目の画像と類似度スコア
    axes[i, 3].imshow(img2_worst)
    axes[i, 3].set_title(f"{images[idx2_worst]} (Similarity: {sim_worst:.2f})", fontsize=10) # 画像名を追加
    axes[i, 3].axis('off')

# 図の中央に垂直の区切り線を追加
# x座標は図の幅に対する相対位置 (0から1)
# y座標は図の高さに対する相対位置 (0から1)
# ここではサブプロット領域のY範囲に合わせて調整
line = lines.Line2D([0.5, 0.5], [0.05, 0.95], transform=fig.transFigure, color='gray', linestyle='--', linewidth=2)
fig.add_artist(line)

plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # タイトルとレイアウトの調整
plt.show()